In [1]:
import pandas as pd
import numpy as np
import config
import os
from scipy.stats import ttest_ind, chi2_contingency
from statsmodels.stats.weightstats import DescrStatsW

from get_data import DataLoader


DataLoader = DataLoader(config.DATA_PATH)
df = DataLoader.load_data()
df_years = DataLoader.filter_years([config.Y1, config.Y2])

In [2]:
# Initialize table
summary2 = []

# 1. WHO Region - Categorical
region_ct = pd.crosstab(df_years["who_region"], df_years["year"])
# print(region_ct)

region_all = df["who_region"].value_counts()
# print(region_all)

chi2, p_region, _, _ = chi2_contingency(region_ct)
# print(region_ct)

# TODO: Include!
summary2.append({
    "Variable": "who_region",
    "Overall": f"{df_years['who_region'].nunique()} categories",
    f"{config.Y1}": region_ct[config.Y1].to_dict(),
    f"{config.Y2}": region_ct[config.Y2].to_dict(),
    "P-value": f"{p_region:.3f}"
})

summary = []
# 2. Continuous variables
cont_vars = ["pm10_concentration", "pm25_concentration", "no2_concentration", "pm25_tempcov"]

for var in cont_vars:
    print(var)
    overall_mean = df_years[var].mean()
    overall_sd = df_years[var].std()

    group_y1 = df_years[df_years["year"] == config.Y1][var].dropna()
    group_y2 = df_years[df_years["year"] == config.Y2][var].dropna()

    tstat, pval = ttest_ind(group_y1, group_y2, equal_var=False)

    summary.append({
        "Variable": var,
        "Overall": f"{overall_mean:.2f} ± {overall_sd:.2f}",
        f"{config.Y1}": f"{group_y1.mean():.2f} ± {group_y1.std():.2f}",
        f"{config.Y2}": f"{group_y2.mean():.2f} ± {group_y2.std():.2f}",
        "P-value": f"{pval:.5f}"
    })

# Convert to DataFrame
summary_table = pd.DataFrame(summary)

print(summary_table)
q1path = os.path.join(config.CUR_DIR, 'data','q1.xlsx')
summary_table.to_excel(q1path, index=False)
latex_table = summary_table.to_latex(index=False)
print(latex_table)

pm10_concentration
pm25_concentration
no2_concentration
pm25_tempcov
             Variable        Overall           2015           2020  P-value
0  pm10_concentration  34.20 ± 36.41  37.76 ± 39.33  30.78 ± 33.03  0.00000
1  pm25_concentration  13.91 ± 15.25  15.11 ± 18.13  12.98 ± 12.53  0.00925
2   no2_concentration  15.90 ± 10.61  18.16 ± 11.52   14.20 ± 9.52  0.00000
3        pm25_tempcov  87.73 ± 22.05  86.04 ± 22.99  89.17 ± 21.12  0.00783
\begin{tabular}{lllll}
\toprule
Variable & Overall & 2015 & 2020 & P-value \\
\midrule
pm10_concentration & 34.20 ± 36.41 & 37.76 ± 39.33 & 30.78 ± 33.03 & 0.00000 \\
pm25_concentration & 13.91 ± 15.25 & 15.11 ± 18.13 & 12.98 ± 12.53 & 0.00925 \\
no2_concentration & 15.90 ± 10.61 & 18.16 ± 11.52 & 14.20 ± 9.52 & 0.00000 \\
pm25_tempcov & 87.73 ± 22.05 & 86.04 ± 22.99 & 89.17 ± 21.12 & 0.00783 \\
\bottomrule
\end{tabular}

